In [55]:
import os
import numpy as np
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter, landscape
import math
from PIL import Image, ImageFont
import matplotlib.pyplot as plt
from PIL import ImageDraw
import time
from sympy import isprime
import gmpy2
import sys
from sklearn.decomposition import PCA

In [22]:
sys.set_int_max_str_digits(1000000)

In [23]:
is_prime = isprime

In [24]:
is_prime(1000)

False

In [25]:
def getBrightnessValues(ALPHABET):
    brightness_values = []
    for digit in ALPHABET:
        # Create an image of the digit
        im_digit = Image.new("L", (10,10), color=255)  # White background
        plt_digit = ImageDraw.Draw(im_digit)
        plt_digit.text((0,0), str(digit), fill=0)  # Black digit
        # Compute average brightness
        total_brightness = 0
        for x in range(10):
            for y in range(10):
                total_brightness += im_digit.getpixel((x,y))
        avg_brightness = total_brightness / 100
        brightness_values.append((avg_brightness, digit))

    # Normalize and sort by brightness
    brightness_values.sort()  # Sort by first element of each tuple (the brightness)
    # Normalize brightness values to range 0 to 1
    min_brightness = brightness_values[0][0]
    max_brightness = brightness_values[-1][0]
    brightness_values = [((b - min_brightness) / (max_brightness - min_brightness), d) for (b,d) in brightness_values]
    
    return brightness_values

In [26]:
def getCharMatrix(SAMPLE_HEIGHT, SAMPLE_WIDTH, pix_gray, brightness_values, return_gray=False):
    gray_matrix = np.zeros((SAMPLE_HEIGHT, SAMPLE_WIDTH))
    for x in range(SAMPLE_WIDTH):
        for y in range(SAMPLE_HEIGHT):
            gray_matrix[y, x] = pix_gray[x, y] / 255.0  # Normalize to 0 to 1

    char_matrix = np.zeros((SAMPLE_HEIGHT, SAMPLE_WIDTH), dtype=str)
    for x in range(SAMPLE_WIDTH):
        for y in range(SAMPLE_HEIGHT):
            brightness = gray_matrix[y, x]
            # Find the character with the closest brightness
            closest_char = None
            closest_diff = 2.0  # Larger than any possible difference
            for (b, d) in brightness_values:
                diff = abs(b - brightness)
                if diff < closest_diff:
                    closest_diff = diff
                    closest_char = str(d)
            char_matrix[y, x] = closest_char

    if return_gray:
        return char_matrix, gray_matrix

    return char_matrix

In [27]:
alphabets = {
    "numeric": "0123456789",
    "hexadecimal": "0123456789ABCDEF",
    "alphanumeric": "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    "alphabetical": "ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    "keyboardsmash": "1234567890!@#$%^&*()qwertyuiopasdfghjklzxcvbnmQWERTYUIOPASDFGHJKLZXCVBNM"
}

In [56]:

def pca_grayscale(img: Image.Image, levels: int | None = None) -> Image.Image:
    # Ensure RGB, drop alpha if present
    rgb = img.convert("RGB")
    arr = np.asarray(rgb, dtype=np.float32) / 255.0

    h, w, _ = arr.shape
    pixels = arr.reshape(-1, 3)

    # PCA projection RGB -> 1D
    gray = PCA(n_components=1).fit_transform(pixels).reshape(h, w)

    # Normalize to [0, 1]
    gray -= gray.min()
    gray /= gray.max() + 1e-8

    # Optional: keep brightness orientation intuitive
    luminance = 0.2126 * arr[..., 0] + 0.7152 * arr[..., 1] + 0.0722 * arr[..., 2]
    if np.corrcoef(gray.ravel(), luminance.ravel())[0, 1] < 0:
        gray = 1.0 - gray

    # Optional discrete brightness levels
    if levels is not None:
        gray = np.round(gray * (levels - 1)) / (levels - 1)

    out = (gray * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(out, mode="L")

In [60]:
def printCharMatrix(char_matrix):
    for y in range(char_matrix.shape[0]):
        line = ""
        for x in range(char_matrix.shape[1]):
            line += char_matrix[y, x]
        print(line)

def charMatrixToPDF(
    char_matrix,
    filename,
    bold=False,
    margin=10,
    bgcolor="#ffffff",
    textcolor="#000000",
    swapped_mask=None,
    swapped_textcolor="#ff0000",
    save_pdf=True,
    save_jpg=True,
    save_txt=True,
    output_suffix="",
    jpg_scale=4,
):
    if filename.endswith(".pdf"):
        filename = filename[:-4]  # Remove .pdf extension
        
    suffix_part = f"_{output_suffix}" if output_suffix else ""
    filename += suffix_part + "_" + str(char_matrix.shape[1]) + "x" + str(char_matrix.shape[0])
    filename += ".pdf"

    # Create outputs directory if it doesn't exist
    os.makedirs("./outputs", exist_ok=True)

    base_name = os.path.splitext(filename)[0]
    filepath = os.path.join("./outputs", filename)
    jpg_path = os.path.join("./outputs", base_name + ".jpg")
    txt_path = os.path.join("./outputs", base_name + ".txt")

    # Calculate font size and spacing
    num_cols = char_matrix.shape[1]
    num_rows = char_matrix.shape[0]

    # Use Courier (built-in monospace font)
    font_name = "Courier-Bold" if bold else "Courier"

    # Start with a reasonable font size
    font_size = 10

    # Courier has a fixed width-to-height ratio of approximately 0.6
    char_width = font_size * 0.6
    line_height = font_size * 0.65

    # Calculate required page dimensions
    page_width = num_cols * char_width + 2 * margin
    page_height = num_rows * line_height + 2 * margin

    c = canvas.Canvas(filepath, pagesize=(page_width, page_height)) if save_pdf else None

    if save_pdf:
        # Set background color
        c.setFillColor(bgcolor)
        c.rect(0, 0, page_width, page_height, fill=1, stroke=0)

        # Set text font
        c.setFont(font_name, font_size)

    # Start from top of page
    y_position = page_height - margin

    rendered_lines = []

    if swapped_mask is None:
        if save_pdf:
            c.setFillColor(textcolor)
        for row in range(num_rows):
            line = ""
            for col in range(num_cols):
                line += char_matrix[row, col]
            rendered_lines.append(line)
            if save_pdf:
                c.drawString(margin, y_position, line)
            y_position -= line_height
    else:
        for row in range(num_rows):
            line = ""
            for col in range(num_cols):
                cell_char = char_matrix[row, col]
                line += cell_char
                if save_pdf:
                    if bool(swapped_mask[row, col]):
                        c.setFillColor(swapped_textcolor)
                    else:
                        c.setFillColor(textcolor)
                    x_position = margin + col * char_width
                    c.drawString(x_position, y_position, cell_char)
            rendered_lines.append(line)
            y_position -= line_height

    if save_pdf:
        c.save()

    if save_jpg:
        jpg_scale = max(1, int(jpg_scale))
        jpg_width = int(math.ceil(page_width * jpg_scale))
        jpg_height = int(math.ceil(page_height * jpg_scale))
        jpg_image = Image.new("RGB", (jpg_width, jpg_height), color=bgcolor)
        jpg_draw = ImageDraw.Draw(jpg_image)
        try:
            jpg_font = ImageFont.truetype("DejaVuSansMono.ttf", font_size * jpg_scale)
        except Exception:
            jpg_font = ImageFont.load_default()

        scaled_margin = margin * jpg_scale
        scaled_char_width = char_width * jpg_scale
        scaled_line_height = line_height * jpg_scale
        y_position = scaled_margin
        for row in range(num_rows):
            if swapped_mask is None:
                jpg_draw.text((scaled_margin, y_position), rendered_lines[row], fill=textcolor, font=jpg_font)
            else:
                x_position = scaled_margin
                for col in range(num_cols):
                    cell_char = char_matrix[row, col]
                    fill = swapped_textcolor if bool(swapped_mask[row, col]) else textcolor
                    jpg_draw.text((x_position, y_position), cell_char, fill=fill, font=jpg_font)
                    x_position += scaled_char_width
            y_position += scaled_line_height
        jpg_image.save(jpg_path, quality=95, subsampling=0)

    if save_txt:
        with open(txt_path, "w", encoding="utf-8") as txt_file:
            txt_file.write("\n".join(rendered_lines))
            txt_file.write("\n")

    if save_pdf:
        print(f"PDF saved to {filepath} (size: {page_width:.1f} x {page_height:.1f} points)")
    if save_jpg:
        print(f"JPG saved to {jpg_path}")
    if save_txt:
        print(f"TXT saved to {txt_path}")

def digitizeImage(
    IMAGE_NAME="cat_phd.png",
    ALPHABET=alphabets["numeric"],
    SAMPLE_WIDTH=100,
    BOLD=False,
    BGCOLOR="#ffffff",
    TEXTCOLOR="#ff0000",
    UPDATE_INTERVAL=5,
    TOP_BUBBLE_PIXELS=20,
    MAX_BUBBLE_FLIPS=5,
    MAX_PRIME_CHECKS=250000,
    SMALL_PRIME_LIMIT=1000,
    PROBABLE_PRIME_ROUNDS=25,
    ENFORCE_PRIMALITY=True,
    SAVE_PDF=True,
    SAVE_JPG=True,
    SAVE_TXT=True,
    QUIET=False,
    GRAYSCALE_METHOD="standard", # Options: "pca" or "standard"
    # Directly open image
    ):
    import io
    from contextlib import redirect_stdout

    def _run_pipeline():
        im = Image.open(f"./inputs/{IMAGE_NAME}")
        width, height = im.size

        SAMPLE_HEIGHT = SAMPLE_WIDTH * height // width
        # Downsample the image
        im_small = im.resize((SAMPLE_WIDTH, SAMPLE_HEIGHT))
        print(f"Output Dimensions: {SAMPLE_WIDTH}(w) x {SAMPLE_HEIGHT}(h) = {SAMPLE_WIDTH * SAMPLE_HEIGHT} chars")

        # Get a matrix of grayscale brightness values for each pixel
        im_gray = None
        if GRAYSCALE_METHOD == "standard":
            im_gray = im_small.convert("L")  # Convert to grayscale
        elif GRAYSCALE_METHOD == "pca":
            im_gray = pca_grayscale(im_small)  # PCA-based grayscale conversion
        else:
            raise ValueError(f"Invalid GRAYSCALE_METHOD: {GRAYSCALE_METHOD}. Use 'standard' or 'pca'.")
        pix_gray = im_gray.load()

        # Get a list of character brightnesses, in increasing order
        brightness_values = getBrightnessValues(ALPHABET)

        # Convert image to character matrix and keep normalized gray levels for bubble scoring
        char_matrix, gray_matrix = getCharMatrix(
            SAMPLE_HEIGHT, SAMPLE_WIDTH, pix_gray, brightness_values, return_gray=True
        )

        swapped_mask = np.zeros_like(char_matrix, dtype=bool)

        if ALPHABET.isnumeric() and ENFORCE_PRIMALITY:
            print("Alphabet is numeric. Enforcing probabilistic primality with fast sieve search.")
            prime_char_matrix, swapped_mask = enforcePrimality(
                char_matrix,
                gray_matrix,
                brightness_values,
                top_n=TOP_BUBBLE_PIXELS,
                max_flips=MAX_BUBBLE_FLIPS,
                max_checks=MAX_PRIME_CHECKS,
                update_interval=UPDATE_INTERVAL,
                small_prime_limit=SMALL_PRIME_LIMIT,
                probable_prime_rounds=PROBABLE_PRIME_ROUNDS,
            )
            prime_found = True
            output_suffix = "P"
        else:
            if ALPHABET.isnumeric():
                print("Alphabet is numeric, but primality enforcement is disabled.")
            else:
                print("Alphabet is non-numeric, so it would not make sense to enforce primality.")
            prime_char_matrix = char_matrix
            prime_found = False
            output_suffix = "C"

        # Render swapped chars in red and unchanged chars in black for numeric mode.
        export_kwargs = {
            "bold": BOLD,
            "margin": 10,
            "bgcolor": BGCOLOR,
            "output_suffix": output_suffix,
            "save_pdf": SAVE_PDF,
            "save_jpg": SAVE_JPG,
            "save_txt": SAVE_TXT,
        }
        if ALPHABET.isnumeric() and ENFORCE_PRIMALITY and prime_found:
            charMatrixToPDF(
                prime_char_matrix,
                IMAGE_NAME.split(".")[0],
                textcolor="#000000",
                swapped_mask=swapped_mask,
                swapped_textcolor="#ff0000",
                **export_kwargs,
            )
        else:
            charMatrixToPDF(
                prime_char_matrix,
                IMAGE_NAME.split(".")[0],
                textcolor=TEXTCOLOR,
                **export_kwargs,
            )

    if QUIET:
        with redirect_stdout(io.StringIO()):
            _run_pipeline()
    else:
        _run_pipeline()

primifyImage = digitizeImage


In [61]:
def charMatrixToBigNumber(char_matrix):
    # Use GMP-backed integers so very large matrices do not hit Python's int limit.
    flat_digits = "".join(char_matrix.reshape(-1).tolist())
    return gmpy2.mpz(flat_digits)

def bigNumberToCharMatrix(big_number, width, height):
    # Keep leading zeros by left-padding to expected length.
    expected_digits = width * height
    s = str(big_number).zfill(expected_digits)
    chars = np.array(list(s), dtype=str)
    return chars.reshape((height, width))

def rankBubblePixels(char_matrix, gray_matrix, brightness_values, top_n=20, exclude_pos=None):
    """
    Rank pixels by how easy it is to switch to a different digit without visual damage.
    Lower penalty means the pixel is more "on the bubble".
    """
    digit_to_brightness = {str(d): float(b) for (b, d) in brightness_values}
    digits = list(digit_to_brightness.keys())
    candidates = []

    for y in range(char_matrix.shape[0]):
        for x in range(char_matrix.shape[1]):
            if exclude_pos is not None and (y, x) == exclude_pos:
                continue

            current_digit = str(char_matrix[y, x])
            if current_digit not in digit_to_brightness:
                continue

            pixel_brightness = float(gray_matrix[y, x])
            current_diff = abs(pixel_brightness - digit_to_brightness[current_digit])

            best_alt = None
            best_alt_diff = None

            for d in digits:
                if d == current_digit:
                    continue
                alt_diff = abs(pixel_brightness - digit_to_brightness[d])
                if best_alt_diff is None or alt_diff < best_alt_diff:
                    best_alt_diff = alt_diff
                    best_alt = d

            if best_alt is None:
                continue

            penalty = best_alt_diff - current_diff
            candidates.append({
                "y": y,
                "x": x,
                "current": current_digit,
                "alternative": best_alt,
                "penalty": float(penalty),
                "current_diff": float(current_diff),
                "alt_diff": float(best_alt_diff),
            })

    candidates.sort(key=lambda c: (c["penalty"], c["alt_diff"]))
    return candidates[:top_n], candidates

def _small_primes(limit=1000):
    sieve = [True] * (limit + 1)
    sieve[0] = False
    sieve[1] = False
    p = 2
    while p * p <= limit:
        if sieve[p]:
            step = p
            start = p * p
            for j in range(start, limit + 1, step):
                sieve[j] = False
        p += 1
    return [n for n in range(2, limit + 1) if sieve[n]]

def _gmp_probable_prime(n, rounds=25):
    # GMP-backed probable primality: 0 composite, 1 probable prime, 2 definitely prime.
    if n < 2:
        return False
    result = gmpy2.is_probab_prime(gmpy2.mpz(n), rounds)
    return result > 0

def findPrimeByBubblePermutations(
    char_matrix,
    gray_matrix,
    brightness_values,
    top_n=20,
    max_flips=5,
    max_checks=250000,
    update_interval=15,
    small_prime_limit=1000,
    probable_prime_rounds=25,
    ):
    """
    Force valid prime-ending digits and then explore bubble-pixel switch permutations.
    Search order prioritizes lower total visual penalty.
    Uses base-plus-delta candidate generation + small-prime sieve + GMP probable primality.
    """
    import itertools

    if char_matrix.size == 0:
        return None, None

    digit_to_brightness = {str(d): float(b) for (b, d) in brightness_values}
    valid_endings = [d for d in ["1", "3", "7", "9"] if d in digit_to_brightness]
    if not valid_endings:
        print("No valid ending digits (1,3,7,9) available in this alphabet.")
        return None, None

    height, width = char_matrix.shape
    total_digits = height * width
    end_pos = (height - 1, width - 1)
    end_idx = total_digits - 1
    end_gray = float(gray_matrix[end_pos[0], end_pos[1]])

    top_candidates, all_candidates = rankBubblePixels(
        char_matrix, gray_matrix, brightness_values, top_n=top_n, exclude_pos=end_pos
    )
    print(f"Scored {len(all_candidates)} candidate pixels; using top {len(top_candidates)} bubble pixels.")

    # Base number and per-position multipliers (10^(digits-1-idx)).
    flat_chars = char_matrix.reshape(-1).tolist()
    base_digits = [int(ch) for ch in flat_chars]
    base_digit_sum = sum(base_digits)
    base_number = int("".join(flat_chars))
    pos_multiplier = [pow(10, total_digits - 1 - idx) for idx in range(total_digits)]

    # Prepare ending options sorted by visual penalty.
    ending_options = []
    for end_digit in valid_endings:
        end_penalty = abs(end_gray - digit_to_brightness[end_digit])
        end_old = base_digits[end_idx]
        end_new = int(end_digit)
        end_delta = (end_new - end_old) * pos_multiplier[end_idx]
        end_digit_sum_delta = end_new - end_old
        ending_options.append((end_digit, end_penalty, end_delta, end_digit_sum_delta))
    ending_options.sort(key=lambda t: t[1])

    # Precompute candidate swap deltas.
    prepared_candidates = []
    for c in top_candidates:
        idx = c["y"] * width + c["x"]
        old_d = int(c["current"])
        new_d = int(c["alternative"])
        prepared_candidates.append({
            "idx": idx,
            "y": c["y"],
            "x": c["x"],
            "old_digit": old_d,
            "new_digit": new_d,
            "delta_n": (new_d - old_d) * pos_multiplier[idx],
            "delta_digit_sum": new_d - old_d,
            "penalty": c["penalty"],
        })

    # Small-prime sieve setup (exclude 2 and 5, handled by last-digit rule).
    small_primes = [p for p in _small_primes(small_prime_limit) if p not in (2, 5)]

    t0 = time.time()
    last_update = -1
    checks = 0
    sieve_rejects = 0

    for end_digit, end_penalty, end_delta, end_digit_sum_delta in ending_options:
        n_end = base_number + end_delta
        digit_sum_end = base_digit_sum + end_digit_sum_delta

        # Try only changing the last digit first.
        checks += 1

        # Very cheap sieve: divisibility by 3 via digit sum.
        if digit_sum_end % 3 != 0:
            passed_small_prime_sieve = True
            for p in small_primes:
                if n_end % p == 0:
                    passed_small_prime_sieve = False
                    break
            if passed_small_prime_sieve and _gmp_probable_prime(n_end, rounds=probable_prime_rounds):
                elapsed = time.time() - t0
                prime_matrix = char_matrix.copy()
                prime_matrix[end_pos[0], end_pos[1]] = end_digit
                swapped_mask = prime_matrix != char_matrix
                print(f"✓ Found probable prime after {checks} checks in {elapsed:.2f} seconds.")
                return prime_matrix, {
                    "checks": checks,
                    "flips": 0,
                    "ending": end_digit,
                    "penalty": float(end_penalty),
                    "sieve_rejects": sieve_rejects,
                }
        else:
            sieve_rejects += 1

        for flips in range(1, max_flips + 1):
            if not prepared_candidates:
                break

            for combo in itertools.combinations(prepared_candidates, flips):
                if checks >= max_checks:
                    print(f"Reached max_checks={max_checks} without finding a probable prime.")
                    return None, {"checks": checks, "sieve_rejects": sieve_rejects}

                checks += 1

                delta_n = end_delta
                delta_digit_sum = end_digit_sum_delta
                total_penalty = end_penalty
                for swap in combo:
                    delta_n += swap["delta_n"]
                    delta_digit_sum += swap["delta_digit_sum"]
                    total_penalty += swap["penalty"]

                candidate_number = base_number + delta_n
                candidate_digit_sum = base_digit_sum + delta_digit_sum

                # Fast modular sieve before expensive probable-prime testing.
                if candidate_digit_sum % 3 == 0:
                    sieve_rejects += 1
                    continue

                passed_small_prime_sieve = True
                for p in small_primes:
                    if candidate_number % p == 0:
                        passed_small_prime_sieve = False
                        break
                if not passed_small_prime_sieve:
                    sieve_rejects += 1
                    continue

                if _gmp_probable_prime(candidate_number, rounds=probable_prime_rounds):
                    elapsed = time.time() - t0
                    prime_matrix = char_matrix.copy()
                    prime_matrix[end_pos[0], end_pos[1]] = end_digit
                    for swap in combo:
                        prime_matrix[swap["y"], swap["x"]] = str(swap["new_digit"])
                    swapped_mask = prime_matrix != char_matrix
                    print(f"✓ Found probable prime after {checks} checks in {elapsed:.2f} seconds.")
                    return prime_matrix, {
                        "checks": checks,
                        "flips": flips,
                        "ending": end_digit,
                        "penalty": float(total_penalty),
                        "sieve_rejects": sieve_rejects,
                    }

                elapsed_int = int(time.time() - t0)
                if update_interval > 0 and elapsed_int > 0 and elapsed_int % update_interval == 0 and elapsed_int != last_update:
                    last_update = elapsed_int
                    print(
                        f"Checked {checks} candidates in {elapsed_int} seconds; "
                        f"sieve rejected {sieve_rejects} so far."
                    )

    return None, {"checks": checks, "sieve_rejects": sieve_rejects}

def enforcePrimality(
    char_matrix,
    gray_matrix,
    brightness_values,
    top_n=20,
    max_flips=5,
    max_checks=250000,
    update_interval=15,
    small_prime_limit=1000,
    probable_prime_rounds=25,
    ):
    """
    Make the digit matrix probable-prime while preserving visual quality.
    Returns (matrix, swapped_mask).
    """
    num_digits = char_matrix.size

    print(f"Original number has {num_digits} digits.")
    print("Searching for probable prime via bubble-pixel permutations...")

    prime_char_matrix, meta = findPrimeByBubblePermutations(
        char_matrix=char_matrix,
        gray_matrix=gray_matrix,
        brightness_values=brightness_values,
        top_n=top_n,
        max_flips=max_flips,
        max_checks=max_checks,
        update_interval=update_interval,
        small_prime_limit=small_prime_limit,
        probable_prime_rounds=probable_prime_rounds,
    )

    if prime_char_matrix is None:
        print("Could not find a probable prime with constrained bubble switches. Returning original matrix.")
        return char_matrix, np.zeros_like(char_matrix, dtype=bool)

    probable_prime_number = charMatrixToBigNumber(prime_char_matrix)
    swapped_mask = prime_char_matrix != char_matrix
    print(
        f"Found probable prime with ending {meta['ending']} after {meta['checks']} checks "
        f"and {meta['flips']} bubble flips."
    )
    print(f"Small-prime sieve rejected {meta['sieve_rejects']} candidates before GMP testing.")
    print(f"GMP probable-prime verdict: {_gmp_probable_prime(probable_prime_number, rounds=probable_prime_rounds)}")
    return prime_char_matrix, swapped_mask

In [73]:
#printCharMatrix(char_matrix)
digitizeImage(
    #IMAGE_NAME="cat_phd.png",
    IMAGE_NAME="stein_am_rhine.jpg",
    ALPHABET=alphabets["numeric"],
    SAMPLE_WIDTH=400,
    BOLD=True,
    BGCOLOR="#ffffff",
    TEXTCOLOR="#000000",
    UPDATE_INTERVAL=15,
    QUIET=False,
    ENFORCE_PRIMALITY=False,
    SAVE_JPG=True,
    SAVE_PDF=False,
    SAVE_TXT=False,
    GRAYSCALE_METHOD="pca"
    )

Output Dimensions: 400(w) x 300(h) = 120000 chars
Alphabet is numeric, but primality enforcement is disabled.
JPG saved to ./outputs/stein_am_rhine_C_400x300.jpg


In [ ]:
"""
  TODO List:

- Windows executable for easy cross-user execution
- Brightness / basic image processing sliders for input image
- Automatically preview using non-prime number
- Ultimately, search for and display the prime version
"""

'\n  TODO List:\n\n- Windows executable for each execution\n- Brightness / basic image processing sliders for input image\n- Automatically preview using non-prime number\n- Ultimately, search for and display the prime version\n'

In [47]:
# Run a timing test that checks sample width vs time for the above query for sample_width ranging from 10 to 40 in increments of 5

widths,times = [],[]

start=10
end=50
delta=2

for sample_width in range(start, end+1, delta):
    print(f"\nTesting with sample width: {sample_width}")
    start_time = time.time()
    digitizeImage(
        IMAGE_NAME="cat_phd.png",
        ALPHABET=alphabets["numeric"],
        SAMPLE_WIDTH=sample_width,
        BOLD=True,
        BGCOLOR="#ffffff",
        TEXTCOLOR="#000000",
        UPDATE_INTERVAL=15,
        TOP_BUBBLE_PIXELS=20,
        MAX_BUBBLE_FLIPS=5,
        MAX_PRIME_CHECKS=250000,
        QUIET=True,
        SAVE_JPG=True,
        SAVE_PDF=False,
        SAVE_TXT=False,
    )
    elapsed_time = time.time() - start_time
    widths.append(sample_width)
    times.append(elapsed_time)
    print(f"Elapsed time for sample width {sample_width}: {elapsed_time:.2f} seconds")


Testing with sample width: 10


NameError: name 'pca_grayscale' is not defined

In [48]:
X = [10,15,20,25,30,35,40]
Y = [0.06,0.10,0.15,0.55,1.32,6.19,48.07]

X,Y = widths,times

X = X[2:]
Y = Y[2:]

# Build fit arrays
x_arr = np.array(X, dtype=float)
y_arr = np.array(Y, dtype=float)

# Fit exponential model: y = A * exp(B*x)
# ln(y) = ln(A) + B*x
if np.any(y_arr <= 0):
    raise ValueError("All Y values must be > 0 for exponential fitting.")

B, logA = np.polyfit(x_arr, np.log(y_arr), 1)
A = float(np.exp(logA))

# Smooth curve for plotting
x_fit = np.linspace(x_arr.min(), x_arr.max(), 300)
y_fit = A * np.exp(B * x_fit)

# State best-fit equation
print(f"Best-fit curve: y = {A:.6g} * exp({B:.6g} * x)")

# Plot data + best-fit curve on log scale
plt.figure(figsize=(10, 6))
plt.plot(X, Y, "o", label="Observed times")
plt.plot(x_fit, y_fit, "-", label="Best fit curve", linewidth=2)

plt.yscale("log")
plt.xlabel("Sample Width")
plt.ylabel("Time (seconds, log scale)")
plt.title("Time vs Sample Width with Best Fit")
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

TypeError: expected non-empty vector for x

In [36]:
def estimate_time(sample_width):
    return A * np.exp(B * sample_width)

width = 70
estimated_time = estimate_time(width)
print(f"Estimated time for sample width {width}: {estimated_time:.2f} seconds = {estimated_time/60:.2f} minutes = {estimated_time/3600:.2f} hours")

Estimated time for sample width 70: 5373.70 seconds = 89.56 minutes = 1.49 hours
